# Layer 8 Topic Sanity Check

Compare medical prompts with coding prompts using GPT-2 Small and its layer-8 residual-stream SAE. Each prompt contributes its top 20 positive features. This is a small sanity check, not evidence that the method generalizes to other prompts.

Two prompt sets are compared with the same rule:
- **original**: 10 hand-written prompts per topic. Topic and phrasing differ together (the coding prompts use "How do I / How can I" more often).
- **matched**: 10 sentence templates, each filled once with a medical and once with a coding subject, so only the topic differs.

A feature is **common** in a group when it appears in at least 5 of its 10 prompts; it is **rare** when it appears in at most 1. A group-enriched feature is common in one group and rare in the other. The verdict calls separation clear only when each group has at least 3 enriched features and the common-feature Jaccard overlap (shared common features divided by the union of common features) is at most 0.25. The rule lives in `eval/sanity.py`.

**Prediction, written before running the matched set:** if phrasing caused most of the original separation, the matched set's enriched features should be mostly topical (medical or code vocabulary) rather than question-shape features. Either outcome is reported.

In [1]:
import inspect
import json
import statistics
import sys
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
project_root = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / "monitor").is_dir() and (path / "eval").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root from the notebook working directory."
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from eval.sanity import compare_groups, feature_counts
from monitor.extract import extract_features
from monitor.labels import get_label
from monitor.model import load_model_and_sae

data_dir = project_root / "eval" / "data"
prompt_sets = {
    "original": json.loads((data_dir / "sanity_prompts.json").read_text(encoding="utf-8")),
    "matched": json.loads((data_dir / "sanity_prompts_matched.json").read_text(encoding="utf-8")),
}
for name, prompts in prompt_sets.items():
    assert set(prompts) == {"medical", "coding"}, name
    assert all(len(group_prompts) == 10 for group_prompts in prompts.values()), name
print(f"Project root: {project_root}")
print(
    "Installed versions: "
    + ", ".join(
        f"{package} {version(package)}" for package in ("sae-lens", "transformer-lens", "torch")
    )
)
print(f"Extraction API: {inspect.signature(extract_features)}")
print(f"Comparison API: {inspect.signature(compare_groups)}")

/Users/sheddy/Desktop/Feature_monitor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/sheddy/Desktop/Feature_monitor
Installed versions: sae-lens 6.54.0, transformer-lens 3.9.0, torch 2.14.1
Extraction API: (prompt: str, k: int = 20) -> list[monitor.extract.FeatureHit]
Comparison API: (counts_a: collections.Counter[int], counts_b: collections.Counter[int]) -> eval.sanity.Separation


## Load and time extraction
Load the model and SAE before starting the clock, so per-prompt timings cover tokenization and feature extraction only.

In [2]:
model, sae = load_model_and_sae()
print(f"Model device: {model.cfg.device}")
print(f"SAE hook: {sae.cfg.metadata.hook_name}")

feature_ids = {name: {} for name in prompt_sets}
timings_seconds = []
for name, prompts in prompt_sets.items():
    for group, group_prompts in prompts.items():
        feature_ids[name][group] = []
        for prompt in group_prompts:
            started = perf_counter()
            hits = extract_features(prompt, k=20)
            timings_seconds.append(perf_counter() - started)
            feature_ids[name][group].append([hit.feature_id for hit in hits])

print(f"Prompts processed: {len(timings_seconds)}")
print(f"Median extraction time: {statistics.median(timings_seconds):.4f} seconds")
print(f"Slowest extraction: {max(timings_seconds):.4f} seconds")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 20978.61it/s]

Loaded pretrained model gpt2-small into HookedTransformer


/Users/sheddy/Desktop/Feature_monitor/.venv/lib/python3.12/site-packages/sae_lens/saes/sae.py:256: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


Model device: cpu
SAE hook: blocks.8.hook_resid_pre


Prompts processed: 40
Median extraction time: 0.0411 seconds
Slowest extraction: 0.0844 seconds


## Features by topic
For each prompt set, the table lists every feature that is common in one topic and rare in the other. Labels come from the local cache or Neuronpedia; lookup failures are shown as `unlabelled`.

In [3]:
results = {}
for name in prompt_sets:
    counts = {group: feature_counts(feature_ids[name][group]) for group in ("medical", "coding")}
    result = compare_groups(counts["medical"], counts["coding"])
    results[name] = result

    rows = [("Medical", f) for f in sorted(result.enriched_a)] + [
        ("Coding", f) for f in sorted(result.enriched_b)
    ]
    table = [
        f"**{name}** prompt set",
        "",
        "| Enriched for | Feature ID | Medical prompts | Coding prompts | Label |",
        "|---|---:|---:|---:|---|",
    ]
    for trend, feature_id in rows:
        label = get_label(feature_id).replace("|", "\\|").replace("\n", " ")
        table.append(
            f"| {trend} | {feature_id} | {counts['medical'][feature_id]} / 10 | "
            f"{counts['coding'][feature_id]} / 10 | {label} |"
        )
    if not rows:
        table.append("| No enriched features | | | | |")
    display(Markdown("\n".join(table)))

**original** prompt set

| Enriched for | Feature ID | Medical prompts | Coding prompts | Label |
|---|---:|---:|---:|---|
| Medical | 18171 | 5 / 10 | 0 / 10 | medical terms related to inflammation or clinical conditions |
| Medical | 23838 | 5 / 10 | 1 / 10 | interrogative sentences ending with a question mark |
| Coding | 10590 | 1 / 10 | 5 / 10 | questions starting with "Can I" or "How can I" |
| Coding | 18200 | 1 / 10 | 6 / 10 | instances of the article "a" before a noun |
| Coding | 21629 | 1 / 10 | 5 / 10 | phrases related to instructing or providing guidance on various topics or tasks |
| Coding | 21981 | 1 / 10 | 5 / 10 | questions starting with "How do you" and "How do we" followed by verbs |

**matched** prompt set

| Enriched for | Feature ID | Medical prompts | Coding prompts | Label |
|---|---:|---:|---:|---|
| Medical | 18171 | 7 / 10 | 0 / 10 | medical terms related to inflammation or clinical conditions |

## Verdict
Same rule for both sets: clear requires at least 3 enriched features in each direction and overlap at most 0.25; weak means at least one enriched feature but not clear; not at all means there are no enriched features.

In [4]:
for name, result in results.items():
    shared = len(result.common_a & result.common_b)
    union = len(result.common_a | result.common_b)
    print(f"[{name}] Layer 8 separates the topics {result.verdict}.")
    print(
        f"  {len(result.enriched_a)} medical-enriched, {len(result.enriched_b)} coding-enriched; "
        f"common features {len(result.common_a)} medical, {len(result.common_b)} coding, "
        f"{shared} shared of {union} (Jaccard {result.jaccard:.3f})."
    )

[original] Layer 8 separates the topics weakly.
  2 medical-enriched, 4 coding-enriched; common features 7 medical, 10 coding, 3 shared of 14 (Jaccard 0.214).
[matched] Layer 8 separates the topics weakly.
  1 medical-enriched, 0 coding-enriched; common features 5 medical, 5 coding, 3 shared of 7 (Jaccard 0.429).
